# Run the test suite

Validates the from-scratch `nmt` package on Colab. Run the cells top to bottom.

1. **Locate + install** finds the repo (the folder containing `pyproject.toml`) and installs the package plus the test deps.
2. **Run all** runs the whole suite; or use the per-file / single-file cells below.

`SKIP` on `test_kv_cache` and the MQA/GQA case in `test_attention` is expected (those variants aren't built yet) - not a failure. `PASSED` is good; paste any `FAILED` / `ERROR` output back to Claude.

## 1. Locate the repo + install

In [1]:
!git clone https://github.com/Rishi-Jain-27/hindi-translator.git /content/hindi-translator

Cloning into '/content/hindi-translator'...
remote: Enumerating objects: 451, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 451 (delta 4), reused 11 (delta 4), pack-reused 434 (from 1)
Receiving objects: 100% (451/451), 13.74 MiB | 24.14 MiB/s, done.
Resolving deltas: 100% (226/226), done.


In [2]:
!pip install -e .

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [3]:
!cd /content/hindi-translator && git pull
!cd /content/hindi-translator && pip install -q -e .


Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 30.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 20.8 MB/s eta 0:00:00
  Building editable for nmt (pyproject.toml) ... done


In [ ]:
import os

# If auto-detect fails, set this to your repo path, e.g. "/content/hindi-translator" or a Drive path.
REPO_DIR = "/content/hindi-translator"

d = REPO_DIR or os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
assert os.path.exists(os.path.join(d, "pyproject.toml")), "repo not found - set REPO_DIR above"
os.chdir(d)
print("repo root:", d)

repo root: /content/hindi-translator


In [5]:
!pip install -q -e . pytest sentencepiece

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nmt (pyproject.toml) ... done


## 2. Run the whole suite

In [6]:
!python -m pytest tests -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collected 80 items                                                             

tests/test_attention.py::test_self_attention_shapes PASSED               [  1%]
tests/test_attention.py::test_cross_attention_shapes PASSED              [  2%]
tests/test_attention.py::test_attention_rows_sum_to_one PASSED           [  3%]
tests/test_attention.py::test_masked_positions_get_zero_weight PASSED    [  5%]
tests/test_attention.py::test_no_nan_when_a_full_row_is_masked PASSED    [  6%]
tests/test_attention.py::test_mqa_gqa_head_broadcasting SKIPPED (MQA...) [  7%]
tests/test_beam.py::test_beam1_matches_greedy_single PASSED              [  8%]
tests/test_beam.py::test_beam1_matches_greedy_longer P

## 3. Or run them one at a time

Runs each file separately with a header, so it's obvious which file fails.

In [7]:
%%bash
for f in test_schedule test_loss test_optim test_masks test_attention test_model_forward \
         test_tokenizer test_data_collate test_ema test_checkpoint test_tracking \
         test_profiling test_loop; do
  echo "===== $f ====="
  python -m pytest tests/$f.py -q
done

===== test_schedule =====
.....                                                                    [100%]
5 passed in 0.02s
===== test_loss =====
....                                                                     [100%]
4 passed in 2.05s
===== test_optim =====
....                                                                     [100%]
4 passed in 3.19s
===== test_masks =====

no tests ran in 0.01s
===== test_attention =====
.....s                                                                   [100%]
5 passed, 1 skipped in 2.07s
===== test_model_forward =====
.....                                                                    [100%]
5 passed in 2.02s
===== test_tokenizer =====
...                                                                      [100%]
3 passed in 0.36s
===== test_data_collate =====
...                                                                      [100%]
3 passed in 2.02s
===== test_ema =====
......                                            

## 4. Run a single file

Edit the filename and run.

In [8]:
!python -m pytest tests/test_loop.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collected 2 items                                                              

tests/test_loop.py::test_loss_decreases PASSED                           [ 50%]
tests/test_loop.py::test_runs_with_ema_swa_ckpt PASSED                   [100%]

============================== 2 passed in 4.06s ===============================


## Fallback: if `!` / `%%bash` are blocked

Some VS Code + Colab connections don't pass shell commands through. Use the Python API instead:

In [9]:
import pytest
pytest.main(["tests", "-v"])  # or ["tests/test_loop.py", "-v"] for a single file

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collecting ... collected 0 items / 18 errors

==================================== ERRORS ====================================
___________________ ERROR collecting tests/test_attention.py ___________________
ImportError while importing test module '/content/hindi-translator/tests/test_attention.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_attention.py:7: in <module>
    from nmt.config import ModelConfig
E   ModuleNotFoundError: No module named 'nmt'

<ExitCode.INTERRUPTED: 2>